# Layer 4: RL Agent Training (Regime-Aware PPO)

Trains a **single regime-aware PPO policy** for portfolio rebalancing using the **RAPO-AS-RL** framework.

Unlike earlier per-regime approaches (3 separate policies), this trains one unified policy conditioned on the HMM regime index via the observation space.

## Environment Design

**State Space** (15 dimensions):
```
[w_btc, w_eth, w_cash, regime_idx, mu_btc, mu_eth, sigma, spread, depth, sigma_port, cum_pnl, trend_30d, vol_pct, trend_strength, ofi]
```

| Index | Feature | Description |
|-------|---------|-------------|
| 0 | w_btc | Current BTC weight |
| 1 | w_eth | Current ETH weight |
| 2 | w_cash | Current cash weight |
| 3 | regime_idx | HMM regime (0=Calm, 1=Volatile, 2=Stressed) |
| 4 | mu_btc | LightGBM BTC return forecast |
| 5 | mu_eth | LightGBM ETH return forecast |
| 6 | sigma | Per-regime volatility (A&S calibrated) |
| 7 | spread | Per-regime spread (A&S calibrated) |
| 8 | depth | Per-regime market depth |
| 9 | sigma_port | Rolling 20-bar portfolio volatility |
| 10 | cum_pnl | Cumulative realized PnL |
| 11 | trend_30d | 30-day price momentum |
| 12 | vol_pct | Volatility percentile (regime signal) |
| 13 | trend_strength | Trend strength indicator |
| 14 | ofi | Order Flow Imbalance |

**Action Space** (2 dimensions, continuous):
```
[target_w_btc, target_w_eth]  →  w_cash = 1 - w_btc - w_eth
```
Actions are clamped to valid weight simplex (cash ≥ 5%).

**Reward Function**:
```
r_t = Sharpe(portfolio_return - A&S_cost, 50-bar window) × 0.01
     + churn_penalty (if executed_delta exceeds threshold)
     + drawdown_penalty (if drawdown exceeds 20%)
```

**Training Hyperparameters** (ultra-conservative):
| Param | Value | Rationale |
|-------|-------|-----------|
| net_arch | [32, 32] | Small network prevents overfitting |
| learning_rate | 3e-5 | Conservative LR |
| clip_range | 0.1 | Conservative clipping |
| ent_coef | 0.0 | No entropy bonus (exploit, not explore) |
| log_std_init | -0.5 | Small action variance |
| total_steps | 100,000 | Sufficient for convergence |

**Train/Test Split**: 75% train / 25% test (chronological)

**Output**: `models/rl/ppo_full.zip` — evaluated via `run_backtest.py`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Change to project root
%cd C:/Users/zihan/capstone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import json
import torch
import torch.nn as nn

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from src.layer4_rl.rl_env import RegimePortfolioEnv

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path("data/processed")
MODELS_DIR = Path("models")
RL_MODELS_DIR = MODELS_DIR / "rl"
RL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

ANN_FACTOR = 288 * 365  # Annualization for 5-min bars

## 1. Load Data and Models

In [ ]:
# Load price features
price_df = pd.read_parquet(DATA_DIR / 'price_features.parquet')
if 'timestamp' in price_df.columns:
    price_df = price_df.set_index('timestamp')
price_df.index = pd.to_datetime(price_df.index)
print(f"Price data: {price_df.shape}")
print(f"Range: {price_df.index.min()} → {price_df.index.max()}")

# Load HMM regime labels
regime_path = MODELS_DIR / 'hmm' / 'regime_labels.csv'
regime_labels = pd.read_csv(regime_path, index_col=0).iloc[:, 0]
regime_labels.index = pd.to_datetime(regime_labels.index)
regime_labels_aligned = regime_labels.reindex(price_df.index).fillna('Calm')
print(f"Regime labels: {regime_labels_aligned.value_counts().to_dict()}")

# Load A&S cost models
as_dir = MODELS_DIR / 'as_cost'
as_models = {}
for regime in ['Calm', 'Volatile', 'Stressed']:
    as_models[regime] = joblib.load(as_dir / f'as_cost_{regime.lower()}.pkl')
    print(f"  {regime}: σ={as_models[regime].get('volatility', 0):.4f}, "
          f"s={as_models[regime].get('spread', 0):.4f}, "
          f"depth={as_models[regime].get('depth', 0):.2f}")

# Load LGBM forecasters
lgbm_dir = MODELS_DIR / 'lgbm'
from src.layer4_rl.rl_env import SyntheticForecaster
lgbm_models = {}
for asset in ['BTC', 'ETH']:
    lgbm_models[asset] = {}
    for regime in ['Calm', 'Volatile', 'Stressed']:
        pkl_path = lgbm_dir / f'lgbm_{asset.lower()}_{regime.lower()}.pkl'
        if pkl_path.exists():
            lgbm_models[asset][regime] = joblib.load(pkl_path)
        else:
            lgbm_models[asset][regime] = SyntheticForecaster(regime, asset)
print(f"\nLGBM forecasters loaded for BTC/ETH × Calm/Volatile/Stressed")

## 2. Train/Test Split

In [ ]:
# 75/25 chronological split
split_idx = int(len(price_df) * 0.75)
TRAIN_END = price_df.index[split_idx]

train_mask = price_df.index <= TRAIN_END
test_mask = price_df.index > TRAIN_END

price_train = price_df[train_mask]
price_test = price_df[test_mask]
regime_train = regime_labels_aligned[train_mask]
regime_test = regime_labels_aligned[test_mask]

print(f"Train: {len(price_train):,} bars "
      f"({price_train.index.min().date()} → {price_train.index.max().date()})")
print(f"Test:  {len(price_test):,} bars "
      f"({price_test.index.min().date()} → {price_test.index.max().date()})")

## 3. Compute Observation Normalization (Train Split Only)

**CRITICAL**: Normalization must be computed from training data only to avoid look-ahead bias.

In [ ]:
# Create environment on train split to compute normalization
env_for_norm = RegimePortfolioEnv(
    price_train, regime_train, as_models, lgbm_models
)
obs_mean = env_for_norm._obs_mean
obs_std = env_for_norm._obs_std

print(f"Observation space: {env_for_norm.observation_space}")
print(f"obs_mean shape: {obs_mean.shape}")
print(f"obs_std[4:10] (forecast/market features): {obs_std[4:10].round(6)}")

## 4. Create Training Environment

In [ ]:
# Create environment with pre-computed normalization (no look-ahead)
train_env = RegimePortfolioEnv(
    price_train, regime_train, as_models, lgbm_models,
    obs_mean=obs_mean, obs_std=obs_std
)

# Wrap in SB3 VecEnv
vec_train = DummyVecEnv([lambda: train_env])

print(f"Training environment ready")
print(f"  obs_space: {train_env.observation_space}")
print(f"  action_space: {train_env.action_space}")

## 5. Define PPO Model (Ultra-Conservative Hyperparameters)

In [ ]:
# Ultra-conservative hyperparameters
HPARAMS = {
    'net_arch': [32, 32],
    'lr': 3e-5,
    'clip': 0.1,
    'n_steps': 64,
    'batch_size': 16,
    'max_grad_norm': 0.5,
    'log_std_init': -0.5,
    'total_steps': 100_000,
    'chunk_size': 500,
}

print("Hyperparameters:")
for k, v in HPARAMS.items():
    print(f"  {k}: {v}")

def make_model(vec_env):
    """Create PPO model."""
    model = PPO(
        'MlpPolicy', vec_env,
        learning_rate=HPARAMS['lr'],
        gamma=0.99,
        clip_range=HPARAMS['clip'],
        n_steps=HPARAMS['n_steps'],
        batch_size=HPARAMS['batch_size'],
        max_grad_norm=HPARAMS['max_grad_norm'],
        device='cpu',
        policy_kwargs={
            'net_arch': HPARAMS['net_arch'],
            'activation_fn': nn.ReLU,
        },
        verbose=0,
        seed=42,
        n_epochs=1,
        ent_coef=0.0,
    )
    # Initialize log_std for better exploration
    model.policy.log_std.data = torch.tensor([HPARAMS['log_std_init'], HPARAMS['log_std_init']])
    return model

def check_nan(model):
    """Check if model weights contain NaN."""
    w = model.policy.action_net.weight.data
    return not torch.isfinite(w).all()

## 6. Training Loop

In [ ]:
# Create model
model = make_model(vec_train)
total_timesteps = 0
nan_count = 0
losses = []

print(f"\nTraining PPO: {HPARAMS['total_steps']:,} steps")
print("-" * 50)

while total_timesteps < HPARAMS['total_steps']:
    # Train chunk
    try:
        model.learn(HPARAMS['chunk_size'], progress_bar=False, reset_num_timesteps=True)
        total_timesteps += HPARAMS['chunk_size']
        nan_count = 0
    except (ValueError, RuntimeError) as e:
        if any(x in str(e).lower() for x in ['nan', 'invalid', 'inf']):
            nan_count += 1
            if nan_count > 3:
                print(f"Too many NaN errors, stopping.")
                break
            print(f"NaN at {total_timesteps} steps — reinitializing...")
            del model
            model = make_model(vec_train)
            continue
        raise

    # Check for NaN
    if check_nan(model):
        print(f"NaN detected at {total_timesteps} — reinitializing...")
        del model
        model = make_model(vec_train)
        continue

    # Log progress every 20 chunks
    if (total_timesteps // HPARAMS['chunk_size']) % 20 == 0:
        w = model.policy.action_net.weight.data
        print(f"  {total_timesteps:,} steps | weight_mean={w.mean().item():.6f}")

print(f"\nTraining complete! Total steps: {total_timesteps:,}")

## 7. Save Model and Summary

In [ ]:
# Save model
model_path = RL_MODELS_DIR / 'ppo_full.zip'
model.save(str(model_path))
print(f"Model saved to: {model_path}")

# Save training summary
summary = {
    'total_steps': int(total_timesteps),
    'train_split': {
        'start': str(price_train.index.min().date()),
        'end': str(price_train.index.max().date()),
        'n_bars': int(len(price_train)),
    },
    'test_split': {
        'start': str(price_test.index.min().date()),
        'end': str(price_test.index.max().date()),
        'n_bars': int(len(price_test)),
    },
    'hyperparameters': HPARAMS,
    'pipeline': 'regime-aware single PPO (not per-regime)',
    'obs_space_dims': 15,
}

summary_path = RL_MODELS_DIR / 'training_full_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {summary_path}")

## 8. Quick Evaluation on Training Environment

In [ ]:
# Run a few episodes on training env to sanity-check
def run_episode(env, model, deterministic=True, max_steps=2000):
    obs, _ = env.reset()
    rewards = []
    done = False
    steps = 0
    
    while not done and steps < max_steps:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, done, _, _ = env.step(action)
        rewards.append(reward)
        steps += 1
    
    return np.array(rewards)

print("Evaluating on training environment (3 episodes)...")
ep_sharpes = []
for i in range(3):
    ep_rets = run_episode(train_env, model, deterministic=False)
    if len(ep_rets) >= 2:
        ann_ret = ep_rets.mean() * ANN_FACTOR
        ann_vol = ep_rets.std() * np.sqrt(ANN_FACTOR)
        sharpe = ann_ret / (ann_vol + 1e-8)
        ep_sharpes.append(sharpe)
        print(f"  Episode {i+1}: steps={len(ep_rets)}, Sharpe={sharpe:.2f}")

if ep_sharpes:
    print(f"\nMean train Sharpe: {np.mean(ep_sharpes):.2f}")
print("\nNote: Train Sharpe is overfitted. See run_backtest.py for test evaluation.")

## 9. Final Notes

**Full evaluation**: Run `python run_backtest.py` to compare RL vs Flat vs A&S+CVaR on the test period.

**Key findings from current pipeline**:
- RL agent converges to near-cash (low Sharpe) under realistic participation-rate execution costs
- Flat(A&S) wins on Sharpe — passive 60/40 BTC/ETH is optimal given A&S cost headwind
- Core finding: Execution costs (~10-52 bps/decision) dominate active rebalancing returns at 5-min frequency